# 回帰分析：演習②

人口密度と住宅面積の単回帰を行い、推定した傾きの95％信頼区間が何を意味するかを、シミュレーションで確かめる。

## 1. 今回の問い

人口密度の高い地域ほど、住宅の床面積は小さいのか。そして、そうして推定した傾きにつく95％信頼区間は、何を意味するのか。

前半では、都道府県別の人口密度と住宅面積のデータで単回帰を行い、対数変換や残差を使って関係を調べる。後半では、その傾きの95％信頼区間の意味を、真の直線が分かっている仮想的な状況のシミュレーションで確かめる。

## 2. ライブラリとデータの用意

人口密度と住宅面積は別々のCSVに入っている。都道府県コードで一つの表にまとめ、以降の分析に使う。

データの出典は、総務省統計局『統計でみる都道府県のすがた2026』である。人口密度のCSVには、総面積1平方キロメートル当たり人口密度（2024年、指標コード A01201）と可住地面積1平方キロメートル当たり人口密度（2024年、A01202）を、住宅面積のCSVには、持ち家住宅の1住宅当たり延べ面積（2023年、H0210301）と借家住宅の1住宅当たり延べ面積（2023年、H0210302）を収めている。

https://www.stat.go.jp/data/k-sugata/index.html

In [ ]:
%pip install japanize-matplotlib-jlite -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
import japanize_matplotlib_jlite

In [ ]:
# データ出典: 総務省統計局『統計でみる都道府県のすがた2026』
# https://www.stat.go.jp/data/k-sugata/index.html

density = pd.read_csv("japan_prefecture_density_2024.csv")
housing = pd.read_csv("japan_prefecture_housing_2023.csv")

prefecture_data = density.merge(housing, on=["prefecture_code", "prefecture"])

print("都道府県数:", len(prefecture_data))
prefecture_data.head()

# 第1部　人口密度と住宅面積の関係を分析する

住宅面積には、持ち家と借家の二つの指標がある。

- `owned_home_floor_area_m2`：持ち家住宅の1住宅当たり延べ面積。
- `rented_home_floor_area_m2`：借家住宅の1住宅当たり延べ面積。

説明変数には、総面積1平方キロメートル当たりの人口密度を使用する。

$$
floor\ area_i
=
\beta_0
+
\beta_1 density_i
+
u_i
$$

$i$ は都道府県を表す。持ち家と借家について、別々の単回帰を推定する。

## 3. 変数の分布を確認する

人口密度は、東京都、大阪府、神奈川県などで非常に高い。一方、多くの県は数百人以下である。

平均値、中央値、最大値を比較し、分布の偏りを確認する。

In [ ]:
columns_for_summary = [
    "population_density_per_km2",
    "inhabitable_density_per_km2",
    "owned_home_floor_area_m2",
    "rented_home_floor_area_m2",
]

prefecture_data[columns_for_summary].describe().T

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(
    prefecture_data["population_density_per_km2"],
    bins=12,
    edgecolor="black",
)
plt.title("都道府県別人口密度の分布")
plt.xlabel("人口密度、人／平方キロメートル")
plt.ylabel("都道府県数")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.show()

## 4. 元の尺度で散布図を描く

最初に、人口密度をそのまま横軸に置く。

人口密度の高い少数の都府県が横軸を広く使用するため、多くの県が図の左側に集中する可能性がある。

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    prefecture_data["population_density_per_km2"],
    prefecture_data["owned_home_floor_area_m2"],
)

for _, row in prefecture_data.iterrows():
    if row["prefecture"] in ["東京都", "大阪府", "富山県", "北海道", "沖縄県"]:
        plt.annotate(
            row["prefecture"],
            (
                row["population_density_per_km2"],
                row["owned_home_floor_area_m2"],
            ),
            xytext=(4, 4),
            textcoords="offset points",
        )

plt.title("人口密度と持ち家住宅の床面積")
plt.xlabel("人口密度、人／平方キロメートル")
plt.ylabel("持ち家住宅の延べ面積、平方メートル")
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    prefecture_data["population_density_per_km2"],
    prefecture_data["rented_home_floor_area_m2"],
)

for _, row in prefecture_data.iterrows():
    if row["prefecture"] in ["東京都", "大阪府", "奈良県", "佐賀県"]:
        plt.annotate(
            row["prefecture"],
            (
                row["population_density_per_km2"],
                row["rented_home_floor_area_m2"],
            ),
            xytext=(4, 4),
            textcoords="offset points",
        )

plt.title("人口密度と借家住宅の床面積")
plt.xlabel("人口密度、人／平方キロメートル")
plt.ylabel("借家住宅の延べ面積、平方メートル")
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

### 演習1　元の尺度の散布図を読む

散布図を見て、次を説明しなさい。

1. 持ち家住宅の床面積は、人口密度が高い都道府県ほど大きいか、小さいか。借家についても同じ向きか。
2. 図の右端にある都道府県はどこか。その住宅の床面積は、多くの県と比べて大きいほうか、小さいほうか。
3. 多くの都道府県が図の左側に集まっているのはなぜか。
4. 左側に集まった県どうしの違いは、この散布図から読み取りやすいか。読み取りにくいとすれば、それは横軸のどのような性質によるか。

In [ ]:
# ここに説明をコメントとして入力する


## 5. 人口密度を対数変換する

人口密度のように、最大値が最小値の数十倍から数百倍になる変数では、自然対数を使うと分布を圧縮できる。

$$
log\_density_i
=
\log(density_i)
$$

対数変換後は、人口密度の絶対的な差よりも倍率の違いに注目する。

- 100人から200人への増加。
- 1,000人から2,000人への増加。

どちらも人口密度が2倍になった変化として扱われる。

In [ ]:
prefecture_data["log_population_density"] = np.log(
    prefecture_data["population_density_per_km2"]
)

prefecture_data[
    [
        "prefecture",
        "population_density_per_km2",
        "log_population_density",
    ]
].sort_values("population_density_per_km2").head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    prefecture_data["log_population_density"],
    prefecture_data["owned_home_floor_area_m2"],
)

for _, row in prefecture_data.iterrows():
    if row["prefecture"] in ["東京都", "大阪府", "富山県", "北海道", "沖縄県"]:
        plt.annotate(
            row["prefecture"],
            (
                row["log_population_density"],
                row["owned_home_floor_area_m2"],
            ),
            xytext=(4, 4),
            textcoords="offset points",
        )

plt.title("人口密度の対数と持ち家住宅の床面積")
plt.xlabel("人口密度の自然対数")
plt.ylabel("持ち家住宅の延べ面積、平方メートル")
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    prefecture_data["log_population_density"],
    prefecture_data["rented_home_floor_area_m2"],
)

plt.title("人口密度の対数と借家住宅の床面積")
plt.xlabel("人口密度の自然対数")
plt.ylabel("借家住宅の延べ面積、平方メートル")
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

## 6. 元の尺度と対数尺度で単回帰を推定する

持ち家と借家について、次の二種類の単回帰を推定する。

元の尺度：

$$
floor\ area_i
=
\beta_0
+
\beta_1 density_i
+
u_i
$$

対数尺度：

$$
floor\ area_i
=
\alpha_0
+
\alpha_1 \log(density_i)
+
v_i
$$

決定係数と残差を比較し、どちらの式が都道府県間の関係を表しやすいか検討する。

In [ ]:
def estimate_simple_regression(data, y_column, x_column):
    X = sm.add_constant(data[[x_column]])
    return sm.OLS(data[y_column], X).fit()

owned_raw = estimate_simple_regression(
    prefecture_data,
    "owned_home_floor_area_m2",
    "population_density_per_km2",
)
owned_log = estimate_simple_regression(
    prefecture_data,
    "owned_home_floor_area_m2",
    "log_population_density",
)

rented_raw = estimate_simple_regression(
    prefecture_data,
    "rented_home_floor_area_m2",
    "population_density_per_km2",
)
rented_log = estimate_simple_regression(
    prefecture_data,
    "rented_home_floor_area_m2",
    "log_population_density",
)

In [ ]:
model_comparison = pd.DataFrame({
    "持ち家・元の尺度": {
        "傾き": owned_raw.params["population_density_per_km2"],
        "傾きのp値": owned_raw.pvalues["population_density_per_km2"],
        "決定係数": owned_raw.rsquared,
    },
    "持ち家・対数尺度": {
        "傾き": owned_log.params["log_population_density"],
        "傾きのp値": owned_log.pvalues["log_population_density"],
        "決定係数": owned_log.rsquared,
    },
    "借家・元の尺度": {
        "傾き": rented_raw.params["population_density_per_km2"],
        "傾きのp値": rented_raw.pvalues["population_density_per_km2"],
        "決定係数": rented_raw.rsquared,
    },
    "借家・対数尺度": {
        "傾き": rented_log.params["log_population_density"],
        "傾きのp値": rented_log.pvalues["log_population_density"],
        "決定係数": rented_log.rsquared,
    },
}).T

model_comparison

### 演習2　モデルを比較する

次を説明しなさい。

1. 4本の回帰で傾きはすべて負か。
2. 元の尺度と対数尺度では、どちらの決定係数が高いか。
3. 持ち家と借家では、人口密度で床面積をよりよく説明できるのはどちらか。決定係数を根拠に述べよ。
4. 対数変換によって、人口密度の高い都府県と他の県との横軸上の間隔はどう変わるか。
5. 決定係数だけでモデルを選ぶのではなく、散布図と残差も確認する必要がある理由。

In [ ]:
# ここに説明をコメントとして入力する


## 7. 回帰直線と残差を確認する

対数モデルの当てはめ値と残差を計算する。

- 正の残差：実際の住宅面積が、人口密度から予測される値より大きい。
- 負の残差：実際の住宅面積が、人口密度から予測される値より小さい。

残差の大きい都道府県では、人口密度以外の要因が住宅面積と関係している可能性がある。

In [ ]:
prefecture_data["owned_fitted"] = owned_log.fittedvalues
prefecture_data["owned_residual"] = owned_log.resid
prefecture_data["rented_fitted"] = rented_log.fittedvalues
prefecture_data["rented_residual"] = rented_log.resid

owned_residual_table = prefecture_data[
    [
        "prefecture",
        "population_density_per_km2",
        "owned_home_floor_area_m2",
        "owned_fitted",
        "owned_residual",
    ]
].sort_values("owned_residual")

rented_residual_table = prefecture_data[
    [
        "prefecture",
        "population_density_per_km2",
        "rented_home_floor_area_m2",
        "rented_fitted",
        "rented_residual",
    ]
].sort_values("rented_residual")

display(owned_residual_table.head())
display(owned_residual_table.tail())

display(rented_residual_table.head())
display(rented_residual_table.tail())

In [ ]:
line_data = prefecture_data.sort_values(
    "log_population_density"
)

plt.figure(figsize=(8, 5))
plt.scatter(
    prefecture_data["log_population_density"],
    prefecture_data["owned_home_floor_area_m2"],
    label="実際の値",
)
plt.plot(
    line_data["log_population_density"],
    line_data["owned_fitted"],
    label="回帰直線",
)

plt.title("人口密度の対数と持ち家住宅面積")
plt.xlabel("人口密度の自然対数")
plt.ylabel("持ち家住宅の延べ面積、平方メートル")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

### 演習3　残差を解釈する

残差の表を見て、次を説明しなさい。

1. 持ち家住宅で、残差が最も大きい都道府県と最も小さい都道府県はどこか。それぞれ、実際の床面積は人口密度から予測される値より広いか狭いか。
2. 借家住宅で残差が大きい都道府県は、持ち家で残差が大きい都道府県と一致するか。
3. 人口密度が近い都道府県どうしでも住宅面積が異なるのは、どのような要因によると考えられるか。
4. 残差の大きい都道府県を地方別に見ると、特定の地方に偏っていないか。

In [ ]:
# ここに説明をコメントとして入力する


## 8. 人口密度の定義を変えて結果を比較する

人口密度には、少なくとも二つの定義がある。

1. 総面積1平方キロメートル当たり人口密度。
2. 可住地面積1平方キロメートル当たり人口密度。

山地や森林が多い県では、総面積と可住地面積の差が大きい。どちらを使うかによって、横軸の意味と回帰結果が変わる可能性がある。

可住地面積人口密度を対数変換し、同じ単回帰を推定する。

In [ ]:
prefecture_data["log_inhabitable_density"] = np.log(
    prefecture_data["inhabitable_density_per_km2"]
)

owned_inhabitable = estimate_simple_regression(
    prefecture_data,
    "owned_home_floor_area_m2",
    "log_inhabitable_density",
)
rented_inhabitable = estimate_simple_regression(
    prefecture_data,
    "rented_home_floor_area_m2",
    "log_inhabitable_density",
)

density_definition_comparison = pd.DataFrame({
    "持ち家・総面積人口密度": {
        "傾き": owned_log.params["log_population_density"],
        "決定係数": owned_log.rsquared,
    },
    "持ち家・可住地人口密度": {
        "傾き": owned_inhabitable.params["log_inhabitable_density"],
        "決定係数": owned_inhabitable.rsquared,
    },
    "借家・総面積人口密度": {
        "傾き": rented_log.params["log_population_density"],
        "決定係数": rented_log.rsquared,
    },
    "借家・可住地人口密度": {
        "傾き": rented_inhabitable.params["log_inhabitable_density"],
        "決定係数": rented_inhabitable.rsquared,
    },
}).T

density_definition_comparison

### 演習4　変数の定義による違いを考える

次を説明しなさい。

1. 総面積人口密度と可住地人口密度では、どちらが常に高いか。
2. 持ち家住宅について、どちらの定義の決定係数が高いか。
3. 借家住宅についてはどうか。
4. 北海道や山地の多い県を比較する場合、可住地人口密度を使う利点。
5. 一つの変数名だけを見ず、定義と分母を確認する必要がある理由。

In [ ]:
# ここに説明をコメントとして入力する


# 第2部　信頼区間の意味を確かめる

## 10. 信頼区間をもう一度見る

持ち家住宅の床面積を人口密度の対数で説明する回帰について、傾きの推定値と95％信頼区間を確認する。

In [ ]:
owned_ci = owned_log.conf_int().loc["log_population_density"]

print("傾きの推定値:", owned_log.params["log_population_density"])
print("95%信頼区間 下限:", owned_ci[0])
print("95%信頼区間 上限:", owned_ci[1])

## 11. 95％信頼区間は何を意味するのか

母数である真の傾きは、一つの固定された値である。標本が変われば、変わるのは信頼区間の方である。同じ方法で多数の信頼区間を作ると、その約95％が真の傾きを含むように作られている。

目の前の一つの区間について「真の傾きが95％の確率でこの中にある」と述べるのは、頻度論の信頼区間の説明としては正確ではない。区間を作る手続きが、繰り返しのなかで95％の割合で真の値を捉える、という意味である。

なお、47都道府県は無作為に抽出された標本ではないため、ここでの信頼区間は、その意味を理解するための理想化された設定として扱う。

## 12. シミュレーションで被覆率を確かめる

真の直線が分かっている仮想的な状況を作る。持ち家の対数モデルの推定値を真の値とみなし、人口密度の対数をそのまま説明変数に使って、正規分布の誤差を加えた標本を多数生成する。各標本で傾きを推定して95％信頼区間を作り、真の傾きを含む区間の割合、すなわち被覆率を数える。

In [ ]:
rng = np.random.default_rng(2026)

B = 5000
true_beta0 = owned_log.params["const"]
true_beta1 = owned_log.params["log_population_density"]
true_sigma = np.sqrt(owned_log.scale)

x = prefecture_data["log_population_density"].to_numpy()
x_c = x - x.mean()
sxx = np.sum(x_c ** 2)
n = len(x)

errors = rng.normal(0, true_sigma, size=(B, n))

# 単回帰の傾きの公式を利用したベクトル化計算
beta1_sim = true_beta1 + (errors @ x_c) / sxx

y_sim = true_beta0 + true_beta1 * x[None, :] + errors
y_mean = y_sim.mean(axis=1, keepdims=True)
beta0_sim = y_mean[:, 0] - beta1_sim * x.mean()
fitted_sim = beta0_sim[:, None] + beta1_sim[:, None] * x[None, :]
resid_sim = y_sim - fitted_sim

sigma2_sim = np.sum(resid_sim ** 2, axis=1) / (n - 2)
se_sim = np.sqrt(sigma2_sim / sxx)

critical = stats.t.ppf(0.975, df=n - 2)
lower = beta1_sim - critical * se_sim
upper = beta1_sim + critical * se_sim

coverage = np.mean((lower <= true_beta1) & (true_beta1 <= upper))

print("標本の大きさ:", n)
print("真の傾き:", true_beta1)
print("推定された傾きの平均:", beta1_sim.mean())
print("推定された傾きの標準偏差:", beta1_sim.std(ddof=1))
print("理論上の標準誤差:", true_sigma / np.sqrt(sxx))
print("95%信頼区間の被覆率:", coverage)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(beta1_sim, bins=50, density=True, alpha=0.7)
plt.axvline(true_beta1, linestyle="--", label="真の傾き")
plt.axvline(beta1_sim.mean(), linestyle=":", label="推定値の平均")

plt.title("5000個の標本から得られた傾きの分布")
plt.xlabel("傾きの推定値")
plt.ylabel("密度")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

被覆率は約0.95になる。5000回の繰り返しのうち、95％前後の区間が真の傾きを含んでいる。これが95％信頼区間の意味である。

シミュレーションで得られた傾きの標準偏差は、理論上の標準誤差に近い。標準誤差は、同じ分析を別の標本で繰り返したときに推定値がどの程度動くかを表す量であり、データそのもののばらつきとは異なる。

### 演習5　被覆率を解釈する

1. 標本の大きさ n を大きくすると、傾きの標準誤差と信頼区間の幅はどうなると考えられるか。理由も述べよ。
2. 目の前の一つの信頼区間について「真の傾きが95％の確率でこの中にある」とは言えないのはなぜか。

## 13. 小レポート

次の構成で、400字から600字程度のレポートを作成しなさい。

1. 今回分析した問いは何か
2. 人口密度を対数変換した理由
3. 持ち家住宅の回帰結果（傾きと決定係数）
4. 残差が大きい都道府県から考えられること
5. 傾きの95％信頼区間の意味

### 小レポート

ここにレポートを記入する。

In [ ]:
# 必要に応じて追加の計算や図を作成する
